# AI Engineering Interview Prep
## Section: Multimodal AI

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


---
## 🖼️ Section 11 — Multimodal AI

### Q1. What are Multimodal AI models, and how do they process different types of data?
**A:** Multimodal AI models accept and generate multiple data types (text, image, audio, video, code) in a unified model.

**How they process different modalities:**
1. **Modality-specific encoders** convert each input type to embeddings:
   - Text → token embeddings (standard Transformer tokeniser)
   - Image → patch embeddings (ViT: 16×16 pixel patches → vectors)
   - Audio → spectrogram features → embedding (Whisper approach)
   - Video → keyframe embeddings + temporal encoding
2. **Projection layer** maps all modality embeddings to a shared embedding dimension
3. **Shared Transformer** processes all modality embeddings together via cross-attention
4. **Modality-specific decoders** for output generation (text, image pixels, audio tokens)

Examples: GPT-4V/o (text+image), Gemini 1.5 (text+image+audio+video), LLaVA (open-source vision-language), DALL-E 3 (text→image), Whisper (audio→text).

🏷️ *ExamGenie AI uses Gemini 2.5 Flash as a multimodal model — sends both PDF page images and text descriptions for contextually accurate MCQ generation.*


### Q2. How do vision-language models process images?
**A:** Vision-language models (VLMs) combine a vision encoder with a language model.

**Architecture (LLaVA/InstructBLIP approach):**
1. **Vision encoder (ViT or CLIP):** split image into 16×16 patches → embed each patch → sequence of patch vectors
2. **Projection layer (MLP or Q-Former):** align patch vectors to the LLM's embedding dimension
3. **Concatenation:** patch vectors are prepended to text token embeddings
4. **LLM processes combined sequence:** attention sees both image patches and text tokens
5. **Generation:** LLM generates text response attending to both modalities

**CLIP vision encoder specifically:**
- Trained contrastively: (image, text) pairs pushed close in shared embedding space
- Can ground visual concepts to language ("a cat wearing a hat" → finds matching image region)

**Resolution trade-off:** More patches = better detail = more tokens = longer context = slower/more expensive.

🏷️ *Gemini 2.5 Flash in ExamGenie AI: processes PDF page images (rendered as 1024×1024) — identifies formulas, diagrams, and text simultaneously for accurate question generation.*


### Q3. How does CLIP work, and why is it important for multi-modal AI?
**A:** CLIP (Contrastive Language-Image Pre-training, OpenAI 2021) learns a shared embedding space for text and images using contrastive learning.

**Training:**
- Dataset: 400M (image, text caption) pairs from the internet
- Two encoders: image encoder (ViT) + text encoder (Transformer)
- Objective: push matching (image, text) pairs close; push non-matching pairs far

**Inference result:** text "a dog playing fetch" and an image of a dog playing fetch produce nearby vectors.

**Why it's foundational:**
1. **Zero-shot image classification:** embed class names as text → compare image embedding to all class text embeddings → argmax = predicted class. No task-specific training.
2. **Cross-modal retrieval:** search images with text queries and vice versa
3. **Multimodal RAG:** retrieve relevant images when answering text questions
4. **Building block for VLMs:** CLIP's visual encoder is used in LLaVA, DALL-E, Stable Diffusion


### Q4. What are the key architectures for multi-modal models?
**A:**
**1. Dual encoder (CLIP-style):**
- Separate encoder per modality → shared embedding space
- Best for: retrieval, classification, cross-modal search
- Weakness: no deep cross-modal interaction within the model

**2. Encoder-decoder with cross-attention (Flamingo, BLIP-2):**
- Visual features injected into language model via cross-attention
- Best for: visual question answering, image captioning

**3. Unified sequence (LLaVA, GPT-4V):**
- Image patches as tokens, concatenated with text tokens
- Single Transformer processes everything
- Most flexible; current state-of-the-art

**4. Diffusion + language model (DALL-E 3, Stable Diffusion XL):**
- Text encoder → conditioning signal → diffusion model generates image
- Best for: text-to-image generation

**5. Token-based generation (Chameleon, Unified-IO):**
- Discretise all modalities into tokens → single autoregressive model generates any modality
- Most unified; still emerging


### Q5. How does image generation work with diffusion models?
**A:** Diffusion models (Stable Diffusion, DALL-E 3, Flux) generate images by learning to reverse a gradual noising process.

**Training:**
1. Take a real image
2. Gradually add Gaussian noise over T steps until it's pure noise
3. Train a neural network (U-Net or Transformer) to predict the noise added at each step → learn to denoise

**Inference (generation):**
1. Start with pure random noise
2. Run the denoising network T times (typically 20-50 steps), gradually removing noise
3. Text conditioning (CFG — Classifier-Free Guidance): steer the denoising process toward the text prompt using: `output = unconditioned + w × (conditioned - unconditioned)` where w is guidance scale
4. After T steps, the noise has become a coherent image

**Latent diffusion (Stable Diffusion):** diffuse and denoise in a compressed latent space (8× smaller than pixel space) → much more efficient. Decode latent → pixel image with a VAE at the end.


### Q6. What is text-to-speech (TTS), and what models are used for it?
**A:** TTS converts text to natural-sounding speech audio.

**Pipeline:**
1. **Text analysis:** normalise text (abbreviations, numbers, punctuation) → phonemes
2. **Acoustic model:** phonemes → mel spectrogram (acoustic features)
3. **Vocoder:** mel spectrogram → audio waveform

**State-of-the-art models:**
- **OpenAI TTS:** high quality, fast, proprietary API
- **ElevenLabs:** ultra-realistic voice cloning, commercial API
- **Kokoro-82M:** open-source, runs on CPU, surprisingly good quality
- **XTTS (Coqui):** open-source, voice cloning from 3 seconds of reference audio
- **Microsoft Edge TTS:** free API, many voices, decent quality

**Key quality dimensions:**
- Naturalness (prosody, intonation)
- Intelligibility (clarity)
- Voice consistency
- Emotional expressiveness
- Speaker similarity (for voice cloning)

For production voice AI: ElevenLabs for quality, Kokoro/Coqui for open-source/privacy.


### Q7. How does speech-to-text (Whisper) work?
**A:** Whisper (OpenAI, 2022) is an encoder-decoder Transformer trained on 680K hours of multilingual audio.

**Architecture:**
1. **Log-mel spectrogram** — audio converted to 80-band mel spectrogram (2D frequency-time representation)
2. **Convolutional feature extractor** — two Conv layers extract local features
3. **Transformer encoder** — encodes the full audio context
4. **Transformer decoder** — generates text tokens autoregressively, conditioned on audio encoding

**What Whisper can do:**
- Transcription (audio → text) in 99 languages
- Translation (non-English audio → English text)
- Language identification
- Timestamps (word-level)

**Model sizes:** Tiny (39M), Base (74M), Small (244M), Medium (769M), Large-v3 (1.5B).

**Production tips:** Use whisper.cpp (CPU-optimised C++) or faster-whisper (CTranslate2) for 4-8× faster inference than PyTorch. For real-time: use streaming ASR (Nova-2 by Deepgram, AssemblyAI).


### Q8. What is multi-modal RAG, and how does it differ from text-only RAG?
**A:** Multi-modal RAG retrieves and reasons over mixed-content documents (text + images + tables + charts).

**Key differences from text-only RAG:**

| | Text RAG | Multi-modal RAG |
|--|---------|-----------------|
| Embedding | Text embeddings | CLIP (text+image in same space) or separate embeddings |
| Index | Text chunk index | Multi-index (text chunks + image embeddings) |
| Retrieval | Text similarity | Multi-modal similarity (query→text AND query→image) |
| Context building | Text chunks | Text chunks + relevant images |
| Generation | Text-only LLM | Vision-language model required |

**Architecture options:**
1. **Embed-and-retrieve:** CLIP-embed images + text; retrieve both; pass to VLM
2. **Caption-and-retrieve:** convert images to captions first; standard text RAG on captions
3. **Native multi-modal:** VLM processes images directly (Gemini, GPT-4V)

🏷️ *ExamGenie AI uses option 3: PDF pages rendered as images → Gemini 2.5 Flash processes text + diagrams + formulas natively → generates accurate MCQs even from complex figures.*


### Q9. How do you build a system that processes both images and text?
**A:**
**Architecture:**
```python
from google.generativeai import GenerativeModel
import base64

def process_image_and_text(image_path: str, question: str) -> str:
    model = GenerativeModel("gemini-2.5-flash")
    with open(image_path, "rb") as f:
        image_data = base64.b64encode(f.read()).decode()
    
    response = model.generate_content([
        {"mime_type": "image/png", "data": image_data},
        question
    ])
    return response.text
```

**Production considerations:**
1. **Image preprocessing:** resize to model's expected resolution; normalise
2. **Multi-image handling:** most VLMs support multiple images in one call
3. **Image storage:** store images in S3; pass URLs or base64 to VLM
4. **Cost:** image tokens are expensive (Gemini: 258 tokens per image at 768×768)
5. **Fallback:** if VLM is down, extract text with OCR (Tesseract) and fall back to text-only LLM
6. **Caching:** cache VLM responses for identical images to avoid re-processing


### Q10. What are multi-modal embeddings, and how are they used for cross-modal search?
**A:** Multi-modal embeddings map different data types (text, image, audio) to the same vector space so cross-modal similarity search works.

**Key models:**
- **CLIP:** text + image in same 512-1024 dim space. Most widely used.
- **ImageBind (Meta):** text + image + audio + video + IMU + depth — 6 modalities in one space
- **E5-mistral-instruct + CLIP:** combine text and image embeddings in a shared space

**Cross-modal search use cases:**
1. "Find images of cats" → embed query text → nearest neighbour in image embedding space → return matching images
2. "Find product descriptions similar to this photo" → embed image → search text embeddings
3. "Find videos containing this sound" → embed audio clip → search video keyframe embeddings

**Implementation:**
```python
from sentence_transformers import SentenceTransformer
from PIL import Image

clip_model = SentenceTransformer("clip-ViT-B-32")
text_emb = clip_model.encode("a dog playing fetch")
image_emb = clip_model.encode(Image.open("dog.jpg"))
similarity = cosine_similarity(text_emb, image_emb)  # works across modalities!
```


### Q11. How do you evaluate multi-modal AI systems?
**A:**
**Image captioning evaluation:**
- CIDEr, BLEU-4, METEOR — n-gram overlap with reference captions
- CLIPScore — semantic similarity between image and generated caption (no reference needed)

**Visual question answering (VQA):**
- VQA accuracy — exact match with ground truth answer
- Open-ended VQA: LLM-as-judge comparing generated answer with reference

**Image generation evaluation:**
- FID (Fréchet Inception Distance): distribution similarity between generated and real images
- CLIP score: alignment between text prompt and generated image
- Human evaluation: aesthetics, prompt following, realism

**Multi-modal RAG evaluation:**
- Standard RAGAS faithfulness + context precision
- Plus: image retrieval recall (were the relevant images retrieved?)

**Task-specific benchmarks:**
- MMBench, MME, LLaVA-Bench (VLM benchmark)
- MMMU (multi-discipline multi-modal understanding)
- VQA v2 (visual question answering)

🏷️ *ExamGenie AI: measure MCQ quality — expert teachers rate question accuracy, difficulty calibration, and whether diagram-based questions correctly reference the image.*


### Q12. What are the challenges of real-time multi-modal AI processing?
**A:**
1. **High token cost:** images cost 258-765 tokens each (Gemini), making real-time VLM calls expensive
2. **Latency:** VLM inference is slower than text-only (more tokens, larger models)
3. **Image preprocessing:** resizing, normalising, encoding to base64 adds overhead
4. **Bandwidth:** sending large images (1-5MB each) adds network latency
5. **Synchronisation:** aligning audio, video, and text streams in real-time is complex
6. **Model availability:** VLM APIs have lower rate limits than text-only APIs

**Solutions:**
1. **Resize images aggressively:** 512×512 is often sufficient; reduces tokens 4× vs 1024×1024
2. **Lazy loading:** process image only if text analysis indicates it's needed
3. **Edge preprocessing:** run lightweight image feature extractor on-device; only send features to API
4. **Async processing:** for non-real-time, queue image processing
5. **Caching:** identical images produce identical embeddings — cache aggressively


### Q13. How do you handle video understanding with AI?
**A:** Video = sequence of frames + audio stream. Processing approaches:

**Frame-level (most common):**
1. Extract keyframes (e.g., every 1 second, or scene-change detection)
2. Embed each keyframe with CLIP/ViT
3. Process with a VLM (Gemini 1.5 Pro handles 1hr videos natively)

**Temporal understanding:**
- Video Transformers (TimeSformer, VideoMAE) process frame sequences with spatiotemporal attention
- For action recognition: 3D CNN or Transformer on short video clips

**Audio-visual:**
- Transcribe audio (Whisper) → combine transcript with visual understanding

**Production approach:**
- For < 1 hour: Gemini 1.5 Pro can handle the full video natively (1M token context)
- For > 1 hour: chunk into segments → process each → aggregate summaries
- For real-time: extract frames + transcribe in parallel; combine for analysis

**Tools:** Gemini 1.5 Pro (most capable), GPT-4V (keyframe-based), VideoLLaMA, Video-LLaVA.


### Q14. What is visual question answering (VQA)?
**A:** VQA = given an image and a natural language question, produce a correct answer.

Example: Image of a parking lot. Question: "How many cars are there?" Answer: "7"

**Types:**
- **Closed-ended:** answer is a predefined class or short word ("yes/no", "red", "7")
- **Open-ended:** free-form text answer
- **Multi-hop VQA:** "What colour is the car to the left of the red truck?" — requires spatial reasoning

**Models:**
- Early: VGG + LSTM (features concatenated)
- Current: VLMs (GPT-4V, Gemini, LLaVA) — prompt the question with the image

**Benchmarks:** VQA v2 (1.1M questions), TextVQA (reading text in images), DocVQA (document understanding), GQA (compositional reasoning).

**Applications:** medical image Q&A ("What abnormality is visible in this X-ray?"), product image Q&A, document understanding.


### Q15. What is document understanding, and how do models parse documents with layouts?
**A:** Document understanding = extracting structured information from documents with complex layouts (PDFs, forms, invoices, tables).

**Challenge:** Standard text extraction ignores spatial relationships. A table's meaning depends on row/column structure, not just word order.

**Approaches:**

1. **Layout-aware models:** LayoutLM, LayoutLMv3, DocFormer — input both text tokens AND their bounding box coordinates; model learns to understand positional relationships
2. **Visual document models:** Donut, Nougat — treat the document page as an image → generate structured output directly without OCR as a separate step
3. **Azure Document Intelligence / AWS Textract:** cloud APIs that return text + bounding boxes + table structure
4. **VLM approach (Gemini):** render page as image → ask VLM to extract structured information

**Production choice:**
- Digitally-created PDFs: pdfplumber → table extraction accurate, fast, free
- Scanned/mixed: Azure Document Intelligence (best quality, paid)
- Simple images with text: Tesseract OCR (free, good enough)


### Q16. How do you fine-tune a vision-language model?
**A:** Fine-tuning a VLM (e.g., LLaVA, InstructBLIP) for a specific task:

**Process:**
1. **Prepare dataset:** (image, question, answer) triplets for your task (medical images, product photos, etc.)
2. **Choose what to fine-tune:**
   - Full fine-tuning: all weights (expensive, best quality)
   - LoRA on language model layers (common choice)
   - Visual projection layer only (cheapest, for adapting visual features)
3. **Training setup:**
   - Keep vision encoder frozen (it's already good at encoding images)
   - Fine-tune LoRA on the LLM component for task-specific generation
   - Optionally fine-tune the projection MLP if domain images are very different
4. **Data format:** LLaVA-style conversation format with `<image>` tokens

**Minimum data:** 500-2000 high-quality (image, instruction, response) triplets. Quality >> quantity.

**Hardware:** 7B VLM with QLoRA: single A100 or 2×RTX 4090.


### Q17. What are the latency and cost considerations for multi-modal AI in production?
**A:**
**Token costs (Gemini 2.0 Flash pricing):**
- Text input: $0.075/1M tokens
- Image input: 258 tokens per image at 768px → $0.019 per image (negligible)
- But: images dominate for high-volume document processing

**Latency factors:**
- Image encoding: ~50-200ms additional vs text-only
- More tokens = longer prefill = higher TTFT
- Resolution matters: 512×512 → fewer tokens, faster

**Cost optimisation:**
1. **Resize images** — process at minimum needed resolution (512px often sufficient for text extraction)
2. **Extract text from PDF programmatically first** — only use VLM for pages with complex layouts/images
3. **Cache VLM outputs** — identical documents produce identical responses; cache by image hash
4. **Batch processing** — for non-real-time document processing, batch by night for lower priority pricing
5. **Model routing** — simple image tasks → GPT-4o-mini Vision (cheaper); complex → GPT-4o or Gemini Pro

🏷️ *ExamGenie AI: optimised PDF processing — pages with text-only use pdfplumber (free), pages with diagrams/formulas use Gemini Flash (cheap). ~70% cost reduction vs processing all pages with VLM.*


### Q18. How do you handle multi-modal content moderation?
**A:**
**Text moderation:** classifier for toxicity, hate speech, harmful content (established).

**Image moderation:**
- Google Cloud Vision SafeSearch API: adult, violent, racy content detection
- AWS Rekognition Moderation: 30+ moderation categories
- Custom classifier: fine-tuned ViT on your labelled moderation categories
- Perceptual hashing (pHash): detect exact or near-identical known bad images

**Audio moderation:**
- Transcribe with Whisper → apply text moderation to transcript
- Tone analysis for aggression/distress

**Video moderation:**
- Frame-level image moderation (every N seconds)
- Audio track moderation
- Temporal patterns: violence often correlates with specific audio + visual patterns

**Multi-modal contradiction detection:**
- Text says "safe" but image is violent → flag the combination
- Image is benign but text caption is harmful → flag

**Human review queue:** all auto-flagged content goes to human moderators. Appeals process for false positives.


### Q19. What is text-to-video generation, and what are the state-of-the-art approaches?
**A:** Text-to-video (T2V) generates a temporally coherent video clip from a text description.

**Key challenge:** maintaining temporal consistency — the same subject must look the same across frames; motion must be physically plausible.

**State-of-the-art:**
- **OpenAI Sora (2024):** Video Diffusion Transformer (DiT) operating in latent space. Generates up to 1-minute HD video. Understands physical world simulation.
- **Google Veo 2 / 3:** Google's T2V model; Veo 3 adds audio generation
- **Kling AI (Kuaishou):** high quality motion; commercially available
- **Wan (Alibaba):** strong open-source T2V model
- **CogVideoX (Tsinghua):** open-source DiT-based T2V

**Architecture pattern:** Text encoder → latent video diffusion model (denoise in 3D latent space: height × width × time) → VAE decoder → pixel video.

**Production use:** Still primarily API-based (Sora, Veo); self-hosting needs significant GPU resources (generating 5s video takes minutes on A100).


### Q20. Explain Multimodal Fusion Techniques: Early Fusion vs Late Fusion.
**A:**
**Early Fusion:** Combine raw features from all modalities at the input level, before any deep processing. All modalities are processed together from the start.
- Example: concatenate image pixels and text tokens → feed to a single model
- Pro: model can learn deep cross-modal interactions
- Con: requires all modalities present; harder to train (different feature scales)

**Late Fusion:** Process each modality independently with separate models, then combine at the decision/output level.
- Example: image classifier score + text classifier score → weighted average → final prediction
- Pro: modular; each model can be optimised independently; handles missing modalities gracefully
- Con: no deep cross-modal interaction; combining independent predictions is crude

**Intermediate (Cross-attention) Fusion:** Separate encoders → cross-attention mechanisms between modality representations → shared processing.
- Example: LLaVA — visual features via cross-attention into language model
- Best of both worlds: separate initial encoding + deep fusion
- **Current state-of-the-art approach** for VLMs

🏷️ *Gemini's architecture uses intermediate fusion — vision encoder processes the image, then the resulting patch embeddings attend to language tokens in the shared Transformer.*


### Q21-26: Multi-modal Failure Scenarios

### Q21. Your VLM generates factually incorrect image descriptions. How do you fix it?
**A:** 
1. **Prompt with specific grounding instruction:** "Describe ONLY what is literally visible in the image. Do not infer, assume, or add context beyond what you can directly see."
2. **Add few-shot examples** of correct descriptive style
3. **Self-critique loop:** "Review your description. Is every statement directly observable in the image?"
4. **Domain-specific fine-tuning** if the VLM lacks domain vocabulary (medical, satellite)
5. **Confidence thresholding:** if logprobs for key claims are low, flag for human review

---

### Q22. Your VLM answers single-image questions but fails on multi-page documents. How do you fix it?
**A:**
1. Process pages sequentially with accumulated context from previous pages
2. Generate a page-level summary for each page, then synthesise across pages
3. Extract layout structure first (page numbers, headings, section breaks) → provide as metadata
4. Use Gemini 1.5 Pro which can handle multi-page PDFs natively (1M token context)
5. Hierarchical processing: chunk pages → summarise per chunk → synthesise summaries

---

### Q23. Your multimodal LLM ignores the image and generates descriptions from text alone. How do you fix it?
**A:**
1. **Explicit image reference prompt:** "Looking SPECIFICALLY at the image provided, describe..."
2. **Verify image encoding:** confirm image is actually included in the API request (check the messages array)
3. **Increase image resolution:** larger images have more tokens → less likely to be underweighted
4. **Image-first prompt ordering:** place image before text in the message (models attend more to earlier context)
5. **Use a VLM that's better at visual grounding:** LLaVA-HD, GPT-4V, Gemini are better than generic models

---

### Q24. Your diffusion model ignores precise control requirements in text prompts. How do you improve controllability?
**A:**
1. **ControlNet:** add explicit structural conditioning (edge maps, depth maps, pose skeletons) — the model must follow the structure
2. **GLIGEN:** grounding with bounding boxes — specify exactly where objects should appear
3. **Better prompt engineering:** more specific prompts, CFG scale tuning (higher = more prompt adherence)
4. **IP-Adapter:** condition on a reference image's visual features for style/appearance consistency
5. **Prompt weighting:** emphasise key terms: `"a (red:1.5) ball on (grass:1.2)"`

---

### Q25. Your diffusion model generates sharp but repetitive images. How do you balance quality vs diversity?
**A:**
1. **Increase temperature** in sampling (η in DDIM, sigma in DDPM) — more noise = more diversity
2. **Lower CFG guidance scale** — less text guidance = more creative freedom
3. **Different seeds** for each generation
4. **Use Ancestral sampling (DDPM)** instead of deterministic DDIM for more stochastic outputs
5. **Negative prompts** to push away common repetitive patterns: "boring, generic, repetitive"
6. **Style augmentation** in the prompt: add random artistic styles, lighting conditions, angles

---

### Q26. Your diffusion model takes too long per image. How do you speed up sampling?
**A:**
1. **DDIM / DDPM-S scheduler:** reduce sampling steps from 50 to 20 with minimal quality loss
2. **LCM (Latent Consistency Model):** distilled diffusion that generates in 4-8 steps with high quality
3. **Lightning models (SDXL-Lightning, FLUX.1-schnell):** 4-step distilled variants
4. **Flash Diffusion:** batched sampling for parallel generation
5. **TensorRT optimisation:** NVIDIA TRT-LLM for diffusion models; 2-4× speedup
6. **Lower resolution first:** generate 512px → upscale with super-resolution model
7. **GPU upgrade:** A100/H100 for production diffusion workloads
